# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

All features are engineered strictly from observations prior to the prediction timestamp ($t-1$ or earlier) at the `(client_id, content_id, report_date)` grain, extending the ML-04 data contract (`month=2026-03`, `is_available IS TRUE`) to predict next-day organic clicks (`target_clicks_next_day` at $t+1$).

In [1]:
import os
import duckdb
import numpy as np
import pandas as pd

# 1. DuckDB & Hugging Face Authentication (aligned with w03_data_contract.ipynb)
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN secret is missing. Configure it in your environment or Colab Secrets."

con = duckdb.connect(database=":memory:")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}');")

MONTH = "2026-03"
DATASET_PATH = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"

# 2. Causal Feature Engineering Query (strictly <= t-1; target at t+1)
query_feature_vector = f"""
WITH base_lags AS (
    SELECT
        report_date,
        client_id,
        content_id,
        gsc_impressions,
        gsc_clicks,
        gsc_position,
        ga4_sessions,

        -- Historical trailing windows (strictly t-7 to t-1; excludes t and t+1)
        AVG(gsc_impressions) OVER w_7d AS raw_impressions_7d_avg,
        SUM(gsc_clicks) OVER w_7d AS raw_clicks_7d_sum,
        COUNT(gsc_impressions) OVER w_7d AS history_days_7d,

        -- Short vs. medium-term impression windows for momentum
        AVG(gsc_impressions) OVER w_3d AS raw_impressions_3d_avg,
        AVG(gsc_impressions) OVER w_14d AS raw_impressions_14d_avg,

        -- Previous day lags (t-1)
        LAG(gsc_clicks, 1) OVER w_order AS raw_clicks_lag_1d,
        LAG(gsc_position, 1) OVER w_order AS raw_position_prev_day,

        -- Trailing 7d CTR (t-7 to t-1)
        SUM(gsc_clicks) OVER w_7d / NULLIF(SUM(gsc_impressions) OVER w_7d, 0) AS raw_ctr_7d_avg,

        -- Trailing 7d GA4 sessions (t-7 to t-1)
        AVG(ga4_sessions) OVER w_7d AS raw_ga4_sessions_7d_avg,

        -- Client GA4 connectivity indicator observed prior to prediction
        MAX(CASE WHEN ga4_sessions IS NOT NULL THEN 1 ELSE 0 END) OVER w_7d AS raw_is_ga4_connected,

        -- Calendar categorical feature known at decision time t
        strftime(CAST(report_date AS DATE), '%a') AS day_of_week,

        -- Ground truth label: next-day organic clicks (t+1)
        LEAD(gsc_clicks, 1) OVER w_order AS target_clicks_next_day

    FROM read_parquet('{DATASET_PATH}')
    WHERE is_available IS TRUE
    WINDOW
        w_order AS (PARTITION BY client_id, content_id ORDER BY report_date),
        w_3d AS (
            PARTITION BY client_id, content_id
            ORDER BY report_date
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ),
        w_7d AS (
            PARTITION BY client_id, content_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ),
        w_14d AS (
            PARTITION BY client_id, content_id
            ORDER BY report_date
            ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
        )
)
SELECT *
FROM base_lags
WHERE target_clicks_next_day IS NOT NULL;
"""

df_raw = con.execute(query_feature_vector).df()

# 3. Deterministic Missing Value Handling & Feature Transformations
df_feat = df_raw.copy()

# Explicit missingness indicators prior to imputation
df_feat["has_7d_history"] = (df_feat["history_days_7d"] >= 7).astype(int)
df_feat["is_position_missing"] = df_feat["raw_position_prev_day"].isna().astype(int)

# Numeric fills
df_feat["impressions_7d_avg"] = df_feat["raw_impressions_7d_avg"].fillna(0.0)
df_feat["clicks_7d_sum"] = df_feat["raw_clicks_7d_sum"].fillna(0.0)
df_feat["clicks_lag_1d"] = df_feat["raw_clicks_lag_1d"].fillna(0.0)
df_feat["position_prev_day"] = df_feat["raw_position_prev_day"].fillna(100.0)
df_feat["ctr_7d_avg"] = df_feat["raw_ctr_7d_avg"].fillna(0.0)
df_feat["ga4_sessions_7d_avg"] = df_feat["raw_ga4_sessions_7d_avg"].fillna(0.0)
df_feat["is_ga4_connected"] = df_feat["raw_is_ga4_connected"].fillna(0).astype(int)

# Momentum ratio: 3d trailing impressions vs 14d trailing impressions (1.0 = neutral on cold-start)
denom_14d = df_feat["raw_impressions_14d_avg"].replace(0, np.nan)
df_feat["impressions_momentum_ratio"] = (
    (df_feat["raw_impressions_3d_avg"] / denom_14d)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
    .clip(0.0, 10.0)
)

# 4. Categorical Engineering & Explicit One-Hot Encoding
def bucket_serp_tier(pos: float, is_missing: int) -> str:
    if is_missing == 1 or pos >= 100.0:
        return "unranked"
    if pos <= 3.0:
        return "top_3"
    if pos <= 10.0:
        return "page_1_rest"
    return "page_2_plus"

df_feat["serp_tier_prev_day"] = [
    bucket_serp_tier(p, m)
    for p, m in zip(df_feat["position_prev_day"], df_feat["is_position_missing"])
]

dow_categories = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
tier_categories = ["top_3", "page_1_rest", "page_2_plus", "unranked"]

df_feat["day_of_week"] = pd.Categorical(df_feat["day_of_week"], categories=dow_categories)
df_feat["serp_tier_prev_day"] = pd.Categorical(df_feat["serp_tier_prev_day"], categories=tier_categories)

dummies_dow = pd.get_dummies(df_feat["day_of_week"], prefix="dow", dtype=int)
dummies_tier = pd.get_dummies(df_feat["serp_tier_prev_day"], prefix="tier", dtype=int)

numeric_feature_cols = [
    "impressions_7d_avg",
    "clicks_7d_sum",
    "clicks_lag_1d",
    "position_prev_day",
    "ctr_7d_avg",
    "impressions_momentum_ratio",
    "ga4_sessions_7d_avg",
    "is_ga4_connected",
    "has_7d_history",
    "is_position_missing",
]

X = pd.concat([df_feat[numeric_feature_cols], dummies_dow, dummies_tier], axis=1)
y = df_feat["target_clicks_next_day"].astype(float)
groups_client = df_feat["client_id"]
dates_series = pd.to_datetime(df_feat["report_date"])

FEATURE_COLS = list(X.columns)

print(f"Observed modeling rows (target available): {len(X):,}")
print(f"Distinct clients represented: {groups_client.nunique()}")
print(f"Total engineered features in X: {len(FEATURE_COLS)}")
print("Feature columns:", FEATURE_COLS)

AssertionError: HF_TOKEN secret is missing. Configure it in your environment or Colab Secrets.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Because Google Search Console (GSC) exhibits a 24–48 hour reporting delay, all search and analytics features are capped at $t-1$ (`1 PRECEDING`) rather than same-day $t$, ensuring strict decision-time availability prior to predicting $t+1$.

| Feature / Group | Type & Encoding | Meaning | Missing Value Handling | Available-When? |
| :--- | :--- | :--- | :--- | :--- |
| `impressions_7d_avg` | Float (continuous) | Trailing 7-day mean organic impressions per URL. | Filled with `0.0` on initial cold-start row (`t=0`). | Strictly prior: window `[t-7, t-1]`. |
| `clicks_7d_sum` | Float (continuous) | Trailing 7-day cumulative organic clicks per URL. | Filled with `0.0` when no prior rows exist. | Strictly prior: window `[t-7, t-1]`. |
| `clicks_lag_1d` | Float (continuous) | Organic clicks observed on the preceding day. | Filled with `0.0` on the first observed day of a URL. | Strictly prior: `t-1`. |
| `position_prev_day` | Float (continuous) | Average SERP rank on the preceding day (lower is better). | Filled with sentinel `100.0` (unranked) + `is_position_missing=1`. | Strictly prior: `t-1`. |
| `ctr_7d_avg` | Float (ratio `[0, 1]`) | Trailing 7-day click-through rate (`sum_clicks / sum_impressions`). | `NULLIF(..., 0)` prevents division by zero; filled with `0.0`. | Strictly prior: window `[t-7, t-1]`. |
| `impressions_momentum_ratio` | Float (clipped `[0, 10]`) | Short-term (`3d`) vs. medium-term (`14d`) impression velocity. | Filled with `1.0` (neutral trend) when `14d` baseline is zero/missing. | Strictly prior: `[t-3, t-1]` / `[t-14, t-1]`. |
| `ga4_sessions_7d_avg` | Float (continuous) | Trailing 7-day average GA4 sessions for the content item. | Filled with `0.0` when GA4 is unlinked or inactive. | Strictly prior: window `[t-7, t-1]`. |
| `is_ga4_connected` | Binary (`0/1`) | Indicates whether the client had active GA4 telemetry in `[t-7, t-1]`. | Filled with `0` when missing. | Strictly prior: window `[t-7, t-1]`. |
| `has_7d_history` | Binary (`0/1`) | Flags whether a full 7-day trailing window was observed (`history_days_7d >= 7`). | Deterministic integer (`0` or `1`), no missing values. | Strictly prior: count over `[t-7, t-1]`. |
| `is_position_missing` | Binary (`0/1`) | Flags rows where `position_prev_day` was `NULL` before sentinel imputation. | Deterministic integer (`0` or `1`), no missing values. | Strictly prior: `t-1`. |
| `dow_Mon` .. `dow_Sun` | Categorical (7 one-hot cols) | Day-of-week seasonality (`Mon`–`Sun`) derived from `report_date`. | Fixed 7-category schema; unseen/null defaults to all-zero vector. | Known at decision time `t` (calendar). |
| `tier_top_3` .. `tier_unranked` | Categorical (4 one-hot cols) | Discretized SERP rank tier (`top_3`, `page_1_rest`, `page_2_plus`, `unranked`). | Missing `position_prev_day` routes deterministically to `tier_unranked`. | Strictly prior: derived from `t-1`. |

In [ ]:
# Programmatic integrity audit of the constructed feature matrix X
audit_summary = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values,
    "missing_count": X.isna().sum().values,
    "inf_count": np.isinf(X.to_numpy()).sum(axis=0),
    "mean": X.mean().round(4).values,
    "std": X.std().round(4).values,
    "min": X.min().round(4).values,
    "max": X.max().round(4).values,
})

assert audit_summary["missing_count"].sum() == 0, "QUALITY ERROR: Residual NaNs found in X."
assert audit_summary["inf_count"].sum() == 0, "QUALITY ERROR: Infinite values found in X."

print("[Feature Matrix Integrity Audit — PASSED]")
print(audit_summary.to_string(index=False))

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

To verify that our feature matrix contains zero temporal, target, or group leakage, we evaluate the clean feature vector `X` under a **client-grouped holdout split** (`GroupShuffleSplit`) and run three adversarial leakage stress tests:
1. **Attack 1 (Label-Derived Proxy):** Injects a noisy transformation of `target_clicks_next_day` (`LEAD(gsc_clicks, 1)`).
2. **Attack 2 (Future Window Contamination):** Replaces trailing `PRECEDING` impressions with a centered window (`ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING`) that peeks into $t+1 \dots t+3$.
3. **Attack 3 (Single-Feature Sentinel & Same-Day Check):** Audits every individual column in `X` for suspiciously high standalone predictive $R^2$ and compares against contemporaneous day-$t$ metrics.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

# 1. Client-Grouped Honest Split (prevents same-client memorization across train/val)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=groups_client))

# Deterministic cap on training rows for fast notebook execution while preserving unseen client validation
rng = np.random.RandomState(42)
if len(train_idx) > 100_000:
    train_idx = rng.choice(train_idx, size=100_000, replace=False)

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

# Baseline Honest Model
honest_rf = RandomForestRegressor(n_estimators=25, max_depth=8, random_state=42, n_jobs=-1)
honest_rf.fit(X_train, y_train)
pred_honest = honest_rf.predict(X_val)
r2_honest = r2_score(y_val, pred_honest)
rmse_honest = np.sqrt(mean_squared_error(y_val, pred_honest))

# Attack 1: Direct Label-Derived Leak (t+1 proxy)
X_train_atk1 = X_train.copy()
X_val_atk1 = X_val.copy()
X_train_atk1["leaked_target_proxy"] = y_train * 0.999
X_val_atk1["leaked_target_proxy"] = y_val * 0.999

atk1_rf = RandomForestRegressor(n_estimators=25, max_depth=8, random_state=42, n_jobs=-1)
atk1_rf.fit(X_train_atk1, y_train)
pred_atk1 = atk1_rf.predict(X_val_atk1)
r2_atk1 = r2_score(y_val, pred_atk1)
rmse_atk1 = np.sqrt(mean_squared_error(y_val, pred_atk1))

# Attack 2: Future-Window Contamination (centered window peeking 3 days into the future)
query_centered_leak = f"""
SELECT
    AVG(gsc_impressions) OVER (
        PARTITION BY client_id, content_id
        ORDER BY report_date
        ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
    ) AS leaked_centered_impressions_7d,
    LEAD(gsc_clicks, 1) OVER (
        PARTITION BY client_id, content_id
        ORDER BY report_date
    ) AS target_clicks_next_day
FROM read_parquet('{DATASET_PATH}')
WHERE is_available IS TRUE;
"""
df_centered = con.execute(query_centered_leak).df()
df_centered = df_centered[df_centered["target_clicks_next_day"].notna()].reset_index(drop=True)

X_train_atk2 = X_train.copy()
X_val_atk2 = X_val.copy()
X_train_atk2["leaked_centered_impressions_7d"] = df_centered["leaked_centered_impressions_7d"].iloc[train_idx].fillna(0.0).values
X_val_atk2["leaked_centered_impressions_7d"] = df_centered["leaked_centered_impressions_7d"].iloc[val_idx].fillna(0.0).values

atk2_rf = RandomForestRegressor(n_estimators=25, max_depth=8, random_state=42, n_jobs=-1)
atk2_rf.fit(X_train_atk2, y_train)
pred_atk2 = atk2_rf.predict(X_val_atk2)
r2_atk2 = r2_score(y_val, pred_atk2)
rmse_atk2 = np.sqrt(mean_squared_error(y_val, pred_atk2))

# Attack 3: Single-Feature Leakage Sentinel across all clean features in X
single_feature_r2 = {}
for col in FEATURE_COLS:
    stump = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42, n_jobs=-1)
    stump.fit(X_train[[col]], y_train)
    single_feature_r2[col] = r2_score(y_val, stump.predict(X_val[[col]]))

max_single_feat = max(single_feature_r2, key=single_feature_r2.get)
max_single_r2 = single_feature_r2[max_single_feat]

# Structural SQL & Feature Assertions
sql_body_without_target = query_feature_vector.split("target_clicks_next_day")[0].upper()
assert "LEAD(" not in sql_body_without_target, "LEAKAGE DETECTED: LEAD() found inside feature definitions."
assert "FOLLOWING" not in sql_body_without_target, "LEAKAGE DETECTED: FOLLOWING window found inside feature definitions."
assert max_single_r2 < 0.90, f"LEAKAGE SENTINEL ALERT: Single feature '{max_single_feat}' achieved suspicious R²={max_single_r2:.4f}."

leakage_report = pd.DataFrame([
    {"experiment": "Honest Feature Vector (Grouped Client Val)", "measured_r2": round(r2_honest, 4), "measured_rmse": round(rmse_honest, 4), "verdict": "CLEAN (Decision-time causal)"},
    {"experiment": "Attack 1: Label-Derived Proxy (LEAD t+1)", "measured_r2": round(r2_atk1, 4), "measured_rmse": round(rmse_atk1, 4), "verdict": "LEAKED (Artificially inflated R²)"},
    {"experiment": "Attack 2: Centered Window (3 FOLLOWING)", "measured_r2": round(r2_atk2, 4), "measured_rmse": round(rmse_atk2, 4), "verdict": "LEAKED (Future window contamination)"},
])

print("[Leakage Hunt & Adversarial Validation Summary]")
print(leakage_report.to_string(index=False))
print(f"\n[Single-Feature Sentinel] Highest standalone validation R²: '{max_single_feat}' = {max_single_r2:.4f} (threshold < 0.90 — PASSED)")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **`raw_client_domain` / client names / page URLs (`url`, `page_url`):** Excluded to uphold FlyRank client confidentiality and prevent the model from memorizing domain-level baseline traffic.
* **`query_text_unhashed` / raw search queries:** Excluded due to privacy restrictions on client query logs and extreme high-cardinality noise.
* **`client_id`:** Retained solely as a partition and `GroupShuffleSplit` key, excluded from `X` so the model generalizes to unseen clients rather than memorizing client IDs.
* **`content_id`:** Retained solely as a window partition key, excluded from `X` to prevent per-URL memorization.
* **`report_date`:** Used only to order causal windows, excluded as a raw feature to avoid spurious monotonic calendar drift.
* **`is_available`:** Used strictly as a pre-filter (`WHERE is_available IS TRUE`) per the ML-04 data contract; constant (`True`) after filtering.
* **`target_clicks_next_day` (`LEAD(gsc_clicks, 1)`):** Ground-truth outcome at $t+1$; occurs strictly after the prediction timestamp.
* **`gsc_clicks` (same-day $t$):** Excluded in favor of `clicks_lag_1d` ($t-1$) and trailing windows to avoid GSC 24–48 hour reporting-lag leakage.
* **`gsc_impressions` (same-day $t$):** Excluded in favor of trailing `[t-7, t-1]` and momentum windows to prevent intraday/unfinalized reporting leakage.
* **`gsc_position` (same-day $t$):** Excluded in favor of `position_prev_day` ($t-1$) so rank signals are strictly finalized before prediction.
* **`ga4_sessions` (same-day $t$):** Excluded in favor of `ga4_sessions_7d_avg` (`[t-7, t-1]`) to avoid partial same-day session accumulation.
* **Post-hoc refresh / product action flags (e.g., `refresh_triggered`, `intervention_status`):** Excluded because content refresh actions are operationally triggered *after* observed traffic declines, creating reverse-causality target leakage.

In [ ]:
# Explicit Exclusion Registry & Automated Privacy/Leakage Guardrail
EXCLUDED_FIELDS = {
    "raw_client_domain": "Privacy & memorization risk: exposes client identity and leaks domain-level baseline traffic.",
    "query_text_unhashed": "Privacy & sparsity risk: raw search queries violate FlyRank confidentiality and add extreme cardinality.",
    "client_id": "Grouping key only: excluded from X so models generalize to unseen clients rather than memorizing IDs.",
    "content_id": "Partition key only: excluded from X to prevent URL-level memorization.",
    "report_date": "Ordering key only: raw date excluded to avoid spurious monotonic calendar drift.",
    "is_available": "Contract pre-filter only (WHERE is_available IS TRUE); constant after filtering.",
    "target_clicks_next_day": "Ground-truth label at t+1; occurs strictly after prediction timestamp.",
    "gsc_clicks": "Contemporaneous day-t metric excluded in favor of t-1 lags to avoid GSC 24-48h reporting lag.",
    "gsc_impressions": "Contemporaneous day-t metric excluded in favor of trailing t-7..t-1 windows.",
    "gsc_position": "Contemporaneous day-t metric excluded in favor of t-1 lagged rank.",
    "ga4_sessions": "Contemporaneous day-t metric excluded in favor of trailing t-7..t-1 sessions.",
    "post_hoc_refresh_flags": "Operational action flags recorded after traffic drops induce reverse-causality leakage.",
}

# Guardrail 1: Zero overlap between excluded fields and active model features
overlap = set(EXCLUDED_FIELDS.keys()) & set(FEATURE_COLS)
assert len(overlap) == 0, f"CONTRACT VIOLATION: Excluded fields found in FEATURE_COLS: {overlap}"

# Guardrail 2: Scan feature names for forbidden substrings (domains, URLs, queries, targets, IDs)
forbidden_tokens = ("target", "lead", "future", "following", "domain", "url", "query", "client_id", "content_id")
flagged_names = [
    col for col in FEATURE_COLS
    if any(tok in col.lower() for tok in forbidden_tokens)
]
assert len(flagged_names) == 0, f"PRIVACY/LEAKAGE VIOLATION: Suspicious feature names detected: {flagged_names}"

print("[Exclusion & Privacy Guardrails — PASSED]")
print(f"Verified {len(EXCLUDED_FIELDS)} excluded fields against {len(FEATURE_COLS)} active features with zero overlap.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.